# Training a Self-Attention Neural Network (Learning $W_Q, W_K, W_V$)

Great observation! Up until now, our weight matrices $W_Q, W_K, W_V$ were just initialized randomly. **No learning had happened yet!**

In this notebook, we will build a complete, trainable Neural Network that uses Self-Attention to solve a classification task:
- Sentence 1: **"river bank overflowed"** $\rightarrow$ Class 0 (**Water / Nature**)
- Sentence 2: **"money bank account"** $\rightarrow$ Class 1 (**Finance**)

We will watch $W_Q, W_K, W_V$ learn via **Backpropagation & Gradient Descent**, and see the attention weights shift from random to meaningful!

## Step 1: Vocabulary & Tokenization Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.manual_seed(42)

# Create a mini vocabulary
vocab = {
    "<PAD>": 0,
    "river": 1,
    "bank": 2,
    "overflowed": 3,
    "money": 4,
    "account": 5
}

# Dataset: (Token IDs, Label)
# Label 0 = Nature/Water, Label 1 = Finance
dataset = [
    ([vocab["river"], vocab["bank"], vocab["overflowed"]], 0),  # "river bank overflowed"
    ([vocab["money"], vocab["bank"], vocab["account"]], 1),     # "money bank account"
]

print("Vocabulary:", vocab)

## Step 2: Define Trainable Self-Attention Neural Network

Our model consists of:
1. `nn.Embedding`: Maps token IDs to vectors $X$.
2. `SelfAttention`: Contains trainable parameters $W_Q, W_K, W_V$.
3. `nn.Linear` Classifier: Takes the contextual embedding of `bank` and predicts Class 0 or Class 1.

In [ ]:
class SelfAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, head_dim, num_classes):
        super().__init__()
        # 1. Embedding Layer
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        
        # 2. Trainable Projection Weights WQ, WK, WV
        self.W_q = nn.Linear(emb_dim, head_dim, bias=False)
        self.W_k = nn.Linear(emb_dim, head_dim, bias=False)
        self.W_v = nn.Linear(emb_dim, head_dim, bias=False)
        
        self.head_dim = head_dim
        
        # 3. Output Classification Layer
        self.classifier = nn.Linear(head_dim, num_classes)
        
    def forward(self, input_ids):
        # Lookup Embeddings X: (seq_len, emb_dim)
        X = self.embedding(input_ids)
        
        # Compute Q, K, V
        Q = self.W_q(X)  # (seq_len, head_dim)
        K = self.W_k(X)  # (seq_len, head_dim)
        V = self.W_v(X)  # (seq_len, head_dim)
        
        # Attention scores S = Q @ K^T / sqrt(d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn_weights = F.softmax(scores, dim=-1)  # (seq_len, seq_len)
        
        # Contextual output H = attn_weights @ V
        H = torch.matmul(attn_weights, V)  # (seq_len, head_dim)
        
        # Take contextual embedding of 'bank' (token index 1 in our sentence)
        bank_contextual = H[1]  # (head_dim,)
        
        # Pass through classifier logits
        logits = self.classifier(bank_contextual.unsqueeze(0))  # (1, num_classes)
        
        return logits, attn_weights

## Step 3: Inspect Attention Weights BEFORE Training (Epoch 0)

Before training, $W_Q, W_K, W_V$ are random. Let's see what `bank` pays attention to initially!

In [ ]:
model = SelfAttentionClassifier(vocab_size=len(vocab), emb_dim=16, head_dim=8, num_classes=2)

# Check sentence 1: "river bank overflowed"
inputs1 = torch.tensor(dataset[0][0])
_, attn_before = model(inputs1)

print("=== Attention Weights for 'river bank overflowed' BEFORE Training ===")
print("Row 1 is 'bank' attending to ['river', 'bank', 'overflowed']:")
print(attn_before[1].detach().numpy().round(3))

## Step 4: The Training Loop (Backpropagation & Weight Updates)

We use **CrossEntropyLoss** and **Adam Optimizer** to update all model parameters ($W_Q, W_K, W_V$, Embeddings, and Classifier).

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

print("Training the Self-Attention Network...\n")
for epoch in range(1, 101):
    total_loss = 0.0
    for token_ids, label in dataset:
        inputs = torch.tensor(token_ids)
        target = torch.tensor([label])
        
        # 1. Zero gradients
        optimizer.zero_grad()
        
        # 2. Forward pass
        logits, _ = model(inputs)
        
        # 3. Calculate Loss
        loss = criterion(logits, target)
        
        # 4. Backward pass (Compute gradients for W_Q, W_K, W_V)
        loss.backward()
        
        # 5. Optimizer step (Update weights: W = W - lr * grad)
        optimizer.step()
        
        total_loss += loss.item()
        
    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | Loss: {total_loss:.4f}")

## Step 5: Inspect Attention Weights AFTER Training!

Now that the network has learned to classify the context of `bank`, let's check how the attention weights shifted!

In [ ]:
print("\n================ AFTER TRAINING ================\n")

# Test Sentence 1: "river bank overflowed"
inputs1 = torch.tensor(dataset[0][0])
logits1, attn1 = model(inputs1)
pred1 = torch.argmax(logits1, dim=-1).item()

print("Sentence 1: ['river', 'bank', 'overflowed']")
print(f"Predicted Class: {pred1} (0 = Nature/Water)")
print("Attention Weights for 'bank' (Row 1 -> ['river', 'bank', 'overflowed']):")
print(attn1[1].detach().numpy().round(3))
print("-" * 50)

# Test Sentence 2: "money bank account"
inputs2 = torch.tensor(dataset[1][0])
logits2, attn2 = model(inputs2)
pred2 = torch.argmax(logits2, dim=-1).item()

print("Sentence 2: ['money', 'bank', 'account']")
print(f"Predicted Class: {pred2} (1 = Finance)")
print("Attention Weights for 'bank' (Row 1 -> ['money', 'bank', 'account']):")
print(attn2[1].detach().numpy().round(3))